# Lesson 2: Testing and Metrics

In the previous notebook we saw that LLM outputs vary across runs. Now we tackle the harder question: **how do you measure whether the output is good?**

We'll work with a concrete task — extracting a plot summary and character names from a Sherlock Holmes story — and build up from observation to metrics to automated evaluation.

1. **Run the extraction** — see how outputs differ
2. **Think about metrics** — what makes an extraction "correct"?
3. **LLM-as-judge** — use a second LLM call to score the output
4. **Design your own judge** — exercise

Run cells in order.

In [12]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()

MODEL = "gpt-5.2"

text_path = Path("labs/02_standalone_agents/prompting/sherlock-short.txt")
if not text_path.exists():
    text_path = Path("../prompting/sherlock-short.txt")
if not text_path.exists():
    text_path = Path("prompting/sherlock-short.txt")
story = text_path.read_text()
print(f"Client ready. Story loaded: {len(story):,} characters")

Client ready. Story loaded: 5,146 characters


## Step 1: Extract plot and characters using a Jinja2 prompt template

We load a **Jinja2 prompt template** from `prompting/plot_prompt_template.j2` that extracts the plot summary and character names as JSON. The template receives `story_text` as a variable.

In [13]:
from jinja2 import Template

# Load the Jinja2 prompt template from file
template_path = Path("prompting/plot_prompt_template.j2")
if not template_path.exists():
    template_path = Path("../prompting/plot_prompt_template.j2")
if not template_path.exists():
    template_path = Path("prompting/plot_prompt_template.j2")
prompt_template = Template(template_path.read_text())

# Render the template
rendered_prompt = prompt_template.render(story_text=story)

# Inspect the rendered prompt
print("=== Rendered prompt (first 500 chars) ===")
print(rendered_prompt[:500])
print("...\n")

# Run extraction
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": rendered_prompt}],
    temperature=0.7,
)
raw = r.choices[0].message.content
try:
    result = json.loads(raw)
except json.JSONDecodeError:
    print("Failed to parse JSON, trying to extract JSON from response...", raw)
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    result = json.loads(clean)

print("=== Model output ===")
print(f"Plot:       {result['plot_summary']}")
print(f"Characters: {result['characters']}")

=== Rendered prompt (first 500 chars) ===
You are given a passage from a mystery story. Extract:
1. A concise plot summary (3-5 sentences) capturing the key events and any suspense or intrigue
2. A list of all characters mentioned (main and secondary)

Return valid JSON only, in this format:
{
  "plot_summary": "...",
  "characters": ["Name1", "Name2", ...]
}

Example output:
{
  "plot_summary": "Detective Poirot arrives at a country estate after receiving an urgent letter from the host. During dinner, the lights go out and a scream is 
...

=== Model output ===
Plot:       Dr. Watson reflects that Sherlock Holmes has always regarded Irene Adler as "the" woman, though Holmes claims no romantic feeling for her. Since Watson’s marriage, the two have grown apart, while Holmes remains in Baker Street, alternating between cocaine use and intense investigative work on difficult cases. On March 20, 1888, Watson passes Baker Street after visiting a patient and, seeing Holmes pacing in a bright

### Inspect the output

Let's look at what the model extracted.

In [7]:
# Quick look at what we got
print("Plot summary:")
print(f"  {result['plot_summary']}\n")
print("Characters found:")
for c in result["characters"]:
    print(f"  - {c}")

Plot summary:
  Dr. Watson reflects on Sherlock Holmes’s singular admiration for Irene Adler, the one woman who stands out in Holmes’s memory despite his usual disdain for emotion. After Watson’s marriage has distanced him from Holmes, he hears only scattered reports of Holmes solving far-flung and delicate cases. On March 20, 1888, Watson passes Baker Street, sees Holmes intensely at work, and is drawn to visit him. Holmes greets Watson with restrained warmth and immediately demonstrates his keen observation by deducing details about Watson’s recent life and health.

Characters found:
  - Sherlock Holmes
  - Dr. Watson
  - Irene Adler
  - Trepoff
  - Atkinson brothers


## Step 2: Think about metrics

Before we automate anything, we need to decide **what "good" means**. This is the hardest part of LLM evaluation — and it's a design decision, not a technical one.

### For character extraction (structured)

This is more tractable. If we have a ground-truth list, we can compute:
- **Precision** — of the names the model returned, how many are real characters?
- **Recall** — of the real characters, how many did the model find?
- **F1** — harmonic mean of precision and recall

### For plot summary (free text)

This is harder. A good summary should:
- Cover the **key events** (completeness)
- Not include events that **didn't happen** (faithfulness)
- Be **concise** (no filler)

There's no simple formula for this. Traditional metrics like BLEU or ROUGE compare word overlap with a reference, but they miss meaning — a perfect paraphrase scores poorly. This is where **LLM-as-judge** comes in.

### Character extraction: precision, recall, F1

Let's compute these against a ground-truth list.

In [8]:
# Ground truth: characters actually mentioned in the passage
GROUND_TRUTH_CHARACTERS = {
    "Sherlock Holmes", "Watson", "Irene Adler",
}

def normalize(name):
    """Lowercase, strip titles and first-name-only variants."""
    return name.lower().strip()

def fuzzy_match(predicted, ground_truth):
    """Check if a predicted name matches any ground truth name (substring match)."""
    p = normalize(predicted)
    for gt in ground_truth:
        g = normalize(gt)
        if p in g or g in p:
            return True
    return False

predicted = result["characters"]
tp = sum(1 for p in predicted if fuzzy_match(p, GROUND_TRUTH_CHARACTERS))
precision = tp / len(predicted) if predicted else 0
recall = tp / len(GROUND_TRUTH_CHARACTERS)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f"Predicted: {predicted}")
print(f"  Precision={precision:.2f}  Recall={recall:.2f}  F1={f1:.2f}")

Predicted: ['Sherlock Holmes', 'Dr. Watson', 'Irene Adler', 'Trepoff', 'Atkinson brothers']
  Precision=0.60  Recall=1.00  F1=0.75


## Step 3: LLM-as-judge for the plot summary

For free-text outputs, we can ask **another LLM call** to evaluate the quality. The idea:
1. Give the judge the original text and the model's summary
2. Ask it to score on specific criteria (completeness, faithfulness, conciseness)
3. Require a structured response (JSON with scores and reasoning)

This isn't perfect — the judge is also an LLM with its own biases — but it scales better than manual review and correlates well with human judgments in practice.

In [15]:
# Load the judge prompt template
judge_template_path = Path("prompting/judge_prompt_template.j2")
judge_template = Template(judge_template_path.read_text())

# Render the judge prompt with the original text and the model's summary
rendered_judge = judge_template.render(
    original_text=story[:2000],
    summary=result["plot_summary"],
)

print("Judging the plot summary...\n")
judge_r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": rendered_judge}],
    temperature=0,
)
scores = json.loads(judge_r.choices[0].message.content)

for criterion, data in scores.items():
    print(f"  {criterion:15s} {data['score']}/5 — {data['reason']}")

Judging the plot summary...

  completeness    2/5 — The passage is mostly Watson’s reflection on Holmes’s attitude toward Irene Adler and Holmes’s general aversion to emotion. The summary adds a later scene (Watson’s marriage, Holmes’s cocaine use, a dated visit to Baker Street, and Holmes deducing details about Watson) that is not present in the provided excerpt, so it does not accurately cover the key events of this specific text.
  faithfulness    1/5 — Only the opening idea—Holmes calling Irene Adler “the woman” and not loving her—is supported. The rest (Watson’s marriage and separation from Holmes, cocaine use, March 20, 1888, Watson seeing Holmes pacing, the visit, and Holmes deducing Watson’s weight gain/medical practice) is not in the excerpt and appears invented relative to the given text.
  conciseness     3/5 — It is reasonably compact, but it includes several specific, extraneous plot details (date, pacing in a lit room, deductions about weight gain and medical practice) t

## Step 4: Exercise — design your own judge

The judge prompt above is just one way to evaluate summaries. **Your turn:**

1. Look at the 3 plot summaries above. Do you agree with the judge's scores? Where does it get it wrong?
2. Write your own judge prompt below. You might:
   - Change the criteria (e.g., add "readability" or "captures mystery/tension")
   - Change the scale (binary yes/no instead of 1-5)
   - Add few-shot examples of good vs bad summaries
   - Be more specific about what counts as a "key event" in this passage
3. Run your judge on the same 3 summaries and compare results.

**Bonus:** Can you make the judge *more consistent* across its own runs? (Hint: temperature, structured output, few-shot examples)

In [10]:
# TODO: Write your own judge prompt
MY_JUDGE_PROMPT = """
YOUR JUDGE PROMPT HERE
"""

judge_r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": MY_JUDGE_PROMPT},
        {"role": "user", "content": f"ORIGINAL TEXT:\n{story[:2000]}\n\nSUMMARY:\n{result['plot_summary']}"},
    ],
    temperature=0,
)
print(judge_r.choices[0].message.content)

The excerpt introduces Irene Adler as “the woman” in Sherlock Holmes’s mind—an exceptional figure who stands out from all others despite Holmes’s general contempt for romantic emotion. Watson explains that Holmes is an extraordinarily logical, observant man who views strong feelings as disruptive “grit” in a finely tuned instrument, useful for studying others but dangerous to his own reasoning.

In the summary, Watson adds that after his marriage he has seen less of Holmes and only hears occasional news of Holmes’s difficult cases. On March 20, 1888, Watson happens to pass Baker Street, finds Holmes deeply engaged in work, and goes in. Holmes receives him with controlled warmth and immediately showcases his deductive skill by inferring recent details about Watson’s life and health.


### Jinja2 cheat sheet for prompts

| Syntax | What it does | Example |
|---|---|---|
| `{{ var }}` | Insert a variable | `{{ story_text }}` |
| `{% for x in list %}` | Loop over items | `{% for task in tasks %}` |
| `{% if cond %}` | Conditional block | `{% if output_format == "json" %}` |
| `{# comment #}` | Comment (not rendered) | `{# TODO: add few-shot examples #}` |
| `{{ var \| upper }}` | Filters (transform values) | `{{ name \| title }}` |

**Workflow summary:**
1. Write your prompt in a `.j2` file (keep it in a `prompting/` folder)
2. Load it with `Template(Path("prompting/my_prompt.j2").read_text())`
3. Render with `template.render(var1=value1, var2=value2)`
4. Pass the rendered string to the LLM as the message content